In [1]:
import pandas as pd
import numpy as np

# =========================================================
# 1. LOAD BASE TICKER & SECTOR MAPPING
# =========================================================
# Files: 'List of Quoted Securities and Issued Quantity' & 'GICS - Daily'
quoted_sec = pd.read_csv('List_of_Quoted_Securities.csv')  # Contains: Ticker, Company Name
gics_sectors = pd.read_csv('GICS_Daily.csv')               # Contains: Ticker, Sector

# Merge to get active stocks with their GICS sector assignment
stock_universe = pd.merge(
    quoted_sec[['Ticker']],
    gics_sectors[['Ticker', 'Sector']],
    on='Ticker',
    how='inner'
)

# =========================================================
# 2. MERGE PUBLIC FLOAT (LIQUIDITY FILTER)
# =========================================================
# File: 'Public Holding - Quarterly'
public_holding = pd.read_csv('Public_Holding_Quarterly.csv') # Contains: Ticker, Public_Holding_Percentage

# Convert percentage column if necessary (e.g., 25% -> 0.25)
if public_holding['Public_Holding_Percentage'].max() > 1.0:
    public_holding['Public_Float_Pct'] = public_holding['Public_Holding_Percentage'] / 100.0
else:
    public_holding['Public_Float_Pct'] = public_holding['Public_Holding_Percentage']

stock_universe = pd.merge(
    stock_universe,
    public_holding[['Ticker', 'Public_Float_Pct']],
    on='Ticker',
    how='left'
)

# =========================================================
# 3. MERGE BETA VALUES
# =========================================================
# File: 'Beta Values'
beta_df = pd.read_csv('Beta_Values.csv')  # Contains: Ticker, Beta

stock_universe = pd.merge(
    stock_universe,
    beta_df[['Ticker', 'Beta']],
    on='Ticker',
    how='left'
)

# =========================================================
# 4. CALCULATE EXPECTED RETURN FROM HISTORICAL PRICES
# =========================================================
# File: 'Daily Share Price List: 2021 - 2025'
prices_df = pd.read_csv('Daily_Share_Price_List_2021_2025.csv')

# Ensure Date column is datetime and sorted
prices_df['Date'] = pd.to_datetime(prices_df['Date'])
prices_df = prices_df.sort_values(by=['Ticker', 'Date'])

# Compute daily percentage returns for each ticker
prices_df['Daily_Return'] = prices_df.groupby('Ticker')['Close_Price'].pct_change()

# Calculate Annualized Expected Return (Mean Daily Return * 252 Trading Days)
expected_returns = prices_df.groupby('Ticker')['Daily_Return'].mean() * 252
expected_returns = expected_returns.reset_index()
expected_returns.columns = ['Ticker', 'Expected_Return']

# Merge Expected Returns into stock_universe
stock_universe = pd.merge(
    stock_universe,
    expected_returns,
    on='Ticker',
    how='left'
)

# =========================================================
# 5. CLEAN MISSING DATA
# =========================================================
# Fill missing Beta with market average (1.0) and drop stocks missing return/float data
stock_universe['Beta'] = stock_universe['Beta'].fillna(1.0)
stock_universe = stock_universe.dropna(subset=['Expected_Return', 'Public_Float_Pct'])

print("--- GENERATED REAL STOCK UNIVERSE ---")
print(stock_universe.head())

FileNotFoundError: [Errno 2] No such file or directory: 'List_of_Quoted_Securities.csv'

In [ ]:
# Apply Split Multipliers from 'Sub Division / Share Splits'
splits = pd.read_csv('Sub_Division_Share_Splits.csv') # Contains: Ticker, Split_Ratio

# Merge splits and adjust historical prices before pct_change()
prices_df = pd.merge(prices_df, splits[['Ticker', 'Split_Ratio']], on='Ticker', how='left')
prices_df['Split_Ratio'] = prices_df['Split_Ratio'].fillna(1.0)
prices_df['Adj_Close'] = prices_df['Close_Price'] / prices_df['Split_Ratio']

# Calculate Daily Returns on Adjusted Close
prices_df['Daily_Return'] = prices_df.groupby('Ticker')['Adj_Close'].pct_change()